In [123]:
import os
import pandas as pd
import json
from datetime import datetime
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Load JSON inputs
#params_df = pd.read_json("json_outputs/customer_params_df_clean.json", lines=True)
categories_df = pd.read_json("json_outputs/customer_categories_df_clean.json", lines=True)
regions_df = pd.read_json("json_outputs/customer_regions_df_clean.json", lines=True)
payments_df = pd.read_json("json_outputs/payment_lines_clean.json", lines=True)
rep_df = pd.read_json("json_outputs/representatives_clean.json", lines=True)
customer_df = pd.read_json("json_outputs/customer_df_clean.json", lines=True)[[
    'CUSTOMER_NUMBER', 'CCAT_CODE', 'REGION_CODE', 'REP_CODE',
    'SETTLE_TERMS', 'NORMAL_PAYTERMS', 'DISCOUNT', 'CREDIT_LIMIT'
]]

In [124]:
csv_folder = os.path.join(os.getcwd(), "csv_outputs")
json_folder = os.path.join(os.getcwd(), "json_outputs")

In [125]:
# Load customer master (fact table)
customer_df = pd.read_json("json_outputs/customer_df_clean.json", lines=True)

# Merge rep info into master
customer_df = customer_df.merge(rep_df, on="REP_CODE", how="left")

In [126]:
# Merge core customer data
merged_df = customer_df \
    .merge(regions_df, on="REGION_CODE", how="left") \
    .merge(categories_df, on="CCAT_CODE", how="left")
    
#.merge(params_df[['CUSTOMER_NUMBER', 'PARAMETER', 'PARAMETER_GROUP']], on="CUSTOMER_NUMBER", how="left") \

In [127]:
merged_df.shape

(2657, 19)

In [128]:
merged_df.columns.tolist() 

['CUSTOMER_NUMBER',
 'CCAT_CODE',
 'REGION_CODE',
 'REP_CODE',
 'SETTLE_TERMS',
 'NORMAL_PAYTERMS',
 'DISCOUNT',
 'CREDIT_LIMIT',
 'REP_DESC',
 'COMM_METHOD',
 'COMMISSION',
 'REP_DESC_CLEAN',
 'REP_GROUP',
 'PARAMETER',
 'PARAMETER_GROUP',
 'REGION_DESC',
 'PROVINCE',
 'CCAT_DESC',
 'CCAT_GROUP']

In [129]:
merged_df.drop(columns=[
    "PARAMETER_GROUP"
], inplace=True, errors="ignore")

In [130]:
merged_df.shape

(2657, 18)

In [131]:
merged_df.columns.tolist() 

['CUSTOMER_NUMBER',
 'CCAT_CODE',
 'REGION_CODE',
 'REP_CODE',
 'SETTLE_TERMS',
 'NORMAL_PAYTERMS',
 'DISCOUNT',
 'CREDIT_LIMIT',
 'REP_DESC',
 'COMM_METHOD',
 'COMMISSION',
 'REP_DESC_CLEAN',
 'REP_GROUP',
 'PARAMETER',
 'REGION_DESC',
 'PROVINCE',
 'CCAT_DESC',
 'CCAT_GROUP']

In [ ]:
# Fill all NaN/null values in object columns with "Unknown"
merged_df.loc[:, merged_df.select_dtypes(include=['object']).columns] = merged_df.select_dtypes(include=['object']).fillna("Unknown")

C:\Users\sltha\AppData\Local\Temp\ipykernel_23132\3298131109.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_df[col].fillna("Unknown", inplace=True)


In [133]:
merged_df

,CUSTOMER_NUMBER,CCAT_CODE,REGION_CODE,REP_CODE,SETTLE_TERMS,NORMAL_PAYTERMS,DISCOUNT,CREDIT_LIMIT,REP_DESC,COMM_METHOD,COMMISSION,REP_DESC_CLEAN,REP_GROUP,PARAMETER,REGION_DESC,PROVINCE,CCAT_DESC,CCAT_GROUP
0,AACJ01,21,25b,ZZZ5,0,90,0,999999,Unknown,Unknown,NaN,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown
1,AACJC1,21,25b,ZZZ5,0,120,0,999999,Unknown,Unknown,NaN,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown
2,AACJC2,5,20a,CONS4,0,120,0,999999,CONSIGNMENTS BM,Gross Profit,0.5,CONSIGNMENT,Channel: Consignment,Unknown,Unknown,Unknown,Consignment,Channel: Consignment
3,AADPRG,6,21a,XX,0,120,0,999999,HOUSE CONSIGNMENTS,Gross Profit,0.0,CONSIGNMENT,Channel: Consignment,Unknown,Unknown,Unknown,Advertising Appro,Internal: Advertising
4,AAMI01,41,10a,02,0,120,0,2000,R,Sales,0.5,R,Sales Rep,Unknown,Durban,KwaZulu-Natal,Unknown,Unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2652,ZHAY02,19,4b,03,0,120,0,2000,BJ,Sales,0.5,BJ,Sales Rep,Unknown,Nelspruit / Tzaneen,Mpumalanga,Unknown,Unknown
2653,ZMAU01,37,11a,03,0,120,0,0,BJ,Sales,0.5,BJ,Sales Rep,Unknown,Free State / Lesotho,Free State,Unknown,Unknown
2654,ZNAE01,46,2b,05,0,120,0,30000,RL,Sales,0.5,RL,Sales Rep,Unknown,Krugersdorp / Sun City,North West,Unknown,Unknown
2655,ZNAEOC,5,20a,STAND,0,120,0,999999,STAND-60PC CONSIGNMENT,Gross Profit,0.0,CONSIGNMENT_STANDS,Channel: Consignment,Unknown,Unknown,Unknown,Consignment,Channel: Consignment


In [134]:
merged_df.to_csv(os.path.join(csv_folder, "customer_merged.csv"), index=False)
merged_df.to_json(os.path.join(json_folder, "customer_merged.json"), orient="records", lines=True)